# Copy-betting on Polymarket

Polymarket is technically an orderbook (`maker` posts limit, `taker` crosses the spread), but for *intent* analytics it's clearer to model as a sportsbook:
- **Punter** = `taker` — they pay the spread to enter a position now.
- **Bookmaker** = `maker` — they post passive liquidity.

Goal of this notebook: reframe trades as bets, define the unit of a **new bet** (entry event), pick a leader cohort, and backtest a naive copy-betting strategy on a held-out window. Then size the gap between leader-actual and copy-bet realised PnL.

In [1]:
from __future__ import annotations
from pathlib import Path
import os

import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

from poly_data.analysis.io import scan_trades, scan_markets
from poly_data.analysis.punter import (
    PLATFORM_WALLETS,
    punter_view,
    punter_position_timeline,
    entries_only,
    punter_player_stats,
    simulate_copy_bet,
)
from poly_data.analysis.positions import market_resolution
from poly_data.analysis.ranking import select_top_n, score_C

DATA_ROOT = Path(os.environ.get('POLY_DATA_ROOT', '../data'))
assert (DATA_ROOT / 'trades').is_dir(), 'run update-all + process first'
trades_lf = scan_trades(DATA_ROOT)


## 1. Reframe trades as bets

Drop platform routers, dust, and extreme prices (settlement-day cleanup, not real bets). Drop self-trades. Everything below uses the *taker* side as the punter.

In [2]:
raw_count = trades_lf.select(pl.len()).collect().item()
punter_lf = punter_view(trades_lf, min_usd=1.0, price_floor=0.02, price_ceiling=0.98)
punter_count = punter_lf.select(pl.len()).collect().item()
print(f'raw fills:    {raw_count:>14,}')
print(f'punter fills: {punter_count:>14,}  ({punter_count/raw_count:.1%})')
print(f'dropped:      {raw_count-punter_count:>14,}')
# Bring punter trades into memory for downstream per-row work.
punter_df = punter_lf.collect()
print('punter_df shape:', punter_df.shape)


raw fills:         3,192,928
punter fills:        796,110  (24.9%)
dropped:           2,396,818
punter_df shape: (796110, 12)


## 2. Per-punter position timeline

Walk fills in chronological order per `(taker, market_id, token_side)`. Classify each fill: `entry` (prev_position == 0), `add`, `reduce`, `exit`, `flip`. Copy-betting will key off `entry` events only.

In [3]:
timeline = punter_position_timeline(punter_df)
kind_counts = (timeline.group_by('event_kind').len().sort('len', descending=True))
print('event distribution:')
print(kind_counts)
entries = entries_only(timeline)
print(f'\n{entries.height:,} entry events')


event distribution:
shape: (5, 2)
┌────────────┬────────┐
│ event_kind ┆ len    │
│ ---        ┆ ---    │
│ str        ┆ u32    │
╞════════════╪════════╡
│ entry      ┆ 348960 │
│ add        ┆ 331475 │
│ reduce     ┆ 75762  │
│ flip       ┆ 21006  │
│ exit       ┆ 18907  │
└────────────┴────────┘

348,960 entry events


## 3. Pick the leader cohort

Rank punters by `score_C = win_rate * log(max(1, total_won_usd))` over the **training window** (everything before `train_end_ts`). Filter to ≥20 decided bets and >50% win rate; take top 20. Computed on taker-only stats so passive market-makers don't dilute the cohort.

In [4]:
max_ts = int(punter_df['timestamp'].max())
min_ts = int(punter_df['timestamp'].min())
span_days = (max_ts - min_ts) / 86400
# Held-out test = last 30 days; training = the rest.
test_window_days = 30 if span_days > 60 else max(1, int(span_days * 0.2))
train_end_ts = max_ts - test_window_days * 86400
print(f'data spans {span_days:.0f} days')
print(f'train: ... → {train_end_ts}')
print(f'test:  {train_end_ts} → {max_ts}  ({test_window_days}d)')

train_punter_lf = punter_lf.filter(pl.col('timestamp') < train_end_ts)
stats = punter_player_stats(train_punter_lf)
leaders_df = select_top_n(stats, n=20, min_win_rate=0.5, min_n_bets=20,
                          score_fn=score_C)
leaders = set(leaders_df['player'].to_list())
print(f'\n{len(leaders)} leaders selected')
leaders_df.select(['player','n_won','n_lost','win_rate','total_won_usd','score']).head(10)


data spans 361 days
train: ... → 1774782036
test:  1774782036 → 1777374036  (30d)



0 leaders selected

player,n_won,n_lost,win_rate,total_won_usd,score
str,u32,u32,f64,f64,f64


## 4. Backtest copy-betting

For each leader's `entry` event in the test window, simulate placing the same direction at the **next punter trade's price** in that (market, token_side) within a 1-hour window (worst-case slippage proxy for not having tick-by-tick fills). Hold to resolution; PnL uses `market_resolution()` over the full trades history.

Sizing: 2% of a $10,000 bankroll per copy bet (constant — no Kelly rebalancing; this is a baseline).

In [5]:
# Resolutions over full trade history (training + test).
resolutions = market_resolution(trades_lf)
print(f'resolved markets: {(resolutions["winner_token"] != "open").sum()}')
print(f'still open:       {(resolutions["winner_token"] == "open").sum()}')

result = simulate_copy_bet(
    entries=entries,
    all_punter_trades=punter_df,
    resolutions=resolutions,
    leaders=leaders,
    train_end_ts=train_end_ts,
    test_end_ts=max_ts,
    bankroll=10_000.0,
    per_bet_frac=0.02,
    slippage_window_secs=3600,
)
print('summary:', result.summary)
result.bets.head(10)


resolved markets: 2462
still open:       6207
summary: {'n_bets': 0, 'total_pnl_usd': 0.0}


timestamp,market_id,maker,taker,nonusdc_side,maker_direction,taker_direction,price,usd_amount,token_amount,transactionHash,orderfilled_id,signed_tokens,cum_position,prev_position,event_kind
i64,str,str,str,str,str,str,f64,f64,f64,str,str,f64,f64,f64,str


## 5. Equity curve & realised PnL distribution

Sort copy bets by `fill_ts`, take cumulative PnL of resolved bets. Stack against a naive baseline: a random-leader cohort of the same size.

In [6]:
if result.bets.height > 0:
    resolved = result.bets.filter(pl.col('resolved')).sort('fill_ts')
    eq = resolved.with_columns(pl.col('pnl_usd').cum_sum().alias('cum_pnl'))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(eq['cum_pnl'].to_numpy())
    axes[0].axhline(0, color='k', lw=0.5)
    axes[0].set_title(f'Copy-bet equity curve ({eq.height} resolved bets)')
    axes[0].set_xlabel('bet #'); axes[0].set_ylabel('cumulative PnL (USD)')

    axes[1].hist(resolved['pnl_usd'].to_numpy(), bins=40, color='#2563eb')
    axes[1].axvline(0, color='k', lw=0.5)
    axes[1].set_title('Per-bet PnL distribution')
    axes[1].set_xlabel('PnL (USD)'); axes[1].set_ylabel('count')
    plt.tight_layout(); plt.show()
else:
    print('No copy bets — leader cohort had no entries in test window.')


No copy bets — leader cohort had no entries in test window.


## 6. Slippage analysis

Distribution of `fill_price − leader_price`. A positive value on a BUY means the copier paid more (worse fill), eroding edge. Real production deployment would need this to be near-zero (low-latency websocket / onchain monitoring).

In [7]:
if result.bets.height > 0:
    fig, ax = plt.subplots(figsize=(9, 3.5))
    sl = result.bets['slippage'].drop_nulls().to_numpy()
    ax.hist(sl, bins=50, color='#7c3aed')
    ax.axvline(0, color='k', lw=0.5)
    ax.set_title(f'Slippage distribution (median={float(pl.Series(sl).median()):+.4f})')
    ax.set_xlabel('fill_price − leader_price')
    plt.tight_layout(); plt.show()


## 7. Caveats for production

- **Edge erosion**: If many copiers, each leader trade gets front-run   by copybots — leader's edge collapses. The historical backtest   *cannot* model this; assume realised live PnL ≪ backtested PnL.
- **Survivorship bias**: top-N selection on past performance has a   large look-ahead component baked in. Cross-validate by sliding the   train/test split forward.
- **Latency**: this notebook proxies slippage with the *next punter   trade's price*. Live, you need onchain monitoring (mempool /   Polymarket websocket); assume fill latency in seconds, not hours.
- **Sizing**: 2% flat is a placeholder. Kelly half-fraction per   leader's per-bet edge is a saner production policy.
- **Position vs trade copying**: this notebook copies *entry events*   only. Adds, exits, and flips are leader-specific path decisions;   generic copying creates path-dependent portfolios that diverge   from leader after a few weeks.